# AG2 + Klavis AI Integration

This tutorial demonstrates how to build AI agents using AG2 (formerly AutoGen) with Klavis Strata MCP servers for enhanced functionality.

## Prerequisites

Before we begin, you'll need:

- **OpenAI API key** - Get at [openai.com](https://openai.com/)
- **Klavis API key** - Get at [klavis.ai](https://klavis.ai/)

In [ ]:
# Install the required packages
%pip install -q klavis python-dotenv "ag2[openai,mcp]>=0.11.1"

In [ ]:
import os
import webbrowser

from klavis import Klavis
from klavis.types import McpServerName
from mcp import ClientSession
from mcp.client.streamable_http import streamable_http_client
from autogen.mcp import create_toolkit
from autogen import AssistantAgent, LLMConfig

# Set environment variables
os.environ["OPENAI_API_KEY"] = "YOUR_OPENAI_API_KEY"  # Replace with your actual OpenAI API key
os.environ["KLAVIS_API_KEY"] = "YOUR_KLAVIS_API_KEY"  # Replace with your actual Klavis API key

## Step 1: Create Strata MCP Server

Create a Strata MCP server that combines multiple services (Gmail and Slack) for enhanced agent capabilities.

In [ ]:
klavis_client = Klavis(api_key=os.getenv("KLAVIS_API_KEY"))

# Create a Strata MCP server with Gmail and Slack integrations
response = klavis_client.mcp_server.create_strata_server(
    servers=[McpServerName.GMAIL, McpServerName.SLACK],
    user_id="demo_user"
)

print(f"Strata MCP server created at: {response.strata_server_url}")

# Handle OAuth authorization if needed
if response.oauth_urls:
    for server_name, oauth_url in response.oauth_urls.items():
        webbrowser.open(oauth_url)
        print(f"Opening OAuth authorization for {server_name}: {oauth_url}")
        input(f"Press Enter after completing {server_name} OAuth authorization...")

## Step 2: Create and Run AG2 Agent with MCP Tools

Connect to the Strata MCP server, set up an AG2 AssistantAgent with the available tools, and run it.

In [ ]:
user_query = "Check my latest 5 emails and summarize them in a Slack message to #general"  # Change this query as needed

async with (
    streamable_http_client(response.strata_server_url) as (
        read_stream,
        write_stream,
        _,
    ),
    ClientSession(read_stream, write_stream) as session,
):
    await session.initialize()

    toolkit = await create_toolkit(session=session, use_mcp_resources=False)
    print(f"Available tools: {[tool.name for tool in toolkit.tools]}")

    # Create AG2 AssistantAgent with MCP tools
    assistant = AssistantAgent(
        name="klavis_assistant",
        system_message=(
            "You are a helpful assistant that uses MCP tools to complete tasks. "
            "When calling execute_action, always pass request body parameters via 'body_schema' "
            "and URL query parameters via 'query_params'. "
            "Check get_action_details to determine which parameters go where."
        ),
        llm_config=LLMConfig(
            {
                "model": "gpt-4o-mini",
                "api_key": os.getenv("OPENAI_API_KEY"),
            }
        ),
    )

    toolkit.register_for_llm(assistant)

    print(f"\n{'='*80}\n👤 USER: {user_query}\n{'='*80}\n")

    result = await assistant.a_run(
        message=user_query,
        tools=toolkit.tools,
        max_turns=10,
        user_input=False,
    )
    await result.process()

## Summary

Congratulations! You've successfully created an AG2 agent that can:

1. **Read emails** using the Gmail MCP server
2. **Send Slack messages** using the Slack MCP server
3. **Coordinate multiple services** through Klavis Strata MCP integration

This demonstrates the power of combining AG2's agent framework with Strata MCP server for building sophisticated AI workflows that can interact with multiple external services seamlessly.